In [1]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import Audio, display


PROJECT_PATH = r"E:\Musical Instrument Recognition"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "processed_data"
)

MULTILABEL_PATH = os.path.join(
    PROCESSED_PATH,
    "multilabel_data"
)

STAGE1_MODEL_PATH = os.path.join(
    PROCESSED_PATH,
    "best_cnn_deep.keras"
)

STAGE2_MODEL_PATH = os.path.join(
    MULTILABEL_PATH,
    "best_cnn_multilabel_v2.keras"
)

TESTING_PATH = os.path.join(
    PROJECT_PATH,
    "IRMAS-TestingData"
)


print("Checking model files...")

if os.path.exists(STAGE1_MODEL_PATH):
    print("✓ Stage 1 model found")
else:
    raise FileNotFoundError(
        f"Stage 1 model not found:\n{STAGE1_MODEL_PATH}"
    )

if os.path.exists(STAGE2_MODEL_PATH):
    print("✓ Stage 2 model found")
else:
    raise FileNotFoundError(
        f"Stage 2 model not found:\n{STAGE2_MODEL_PATH}"
    )


stage1_model = tf.keras.models.load_model(
    STAGE1_MODEL_PATH,
    compile=False
)

stage2_model = tf.keras.models.load_model(
    STAGE2_MODEL_PATH,
    compile=False
)


class_names = [
    "cel",
    "cla",
    "flu",
    "gac",
    "gel",
    "org",
    "pia",
    "sax",
    "tru",
    "vio",
    "voi"
]


index_to_label = {
    i: label
    for i, label in enumerate(class_names)
}


label_to_index = {
    label: i
    for i, label in enumerate(class_names)
}


instrument_names = {
    "cel": "Cello",
    "cla": "Clarinet",
    "flu": "Flute",
    "gac": "Acoustic Guitar",
    "gel": "Electric Guitar",
    "org": "Organ",
    "pia": "Piano",
    "sax": "Saxophone",
    "tru": "Trumpet",
    "vio": "Violin",
    "voi": "Voice"
}


final_thresholds = {
    "cel": 0.175,
    "cla": 0.350,
    "flu": 0.400,
    "gac": 0.175,
    "gel": 0.475,
    "org": 0.450,
    "pia": 0.125,
    "sax": 0.225,
    "tru": 0.200,
    "vio": 0.175,
    "voi": 0.125
}


SR = 22050
WINDOW_SECONDS = 3.0
HOP_SECONDS = 1.5

N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

WINDOW_SAMPLES = int(
    SR * WINDOW_SECONDS
)

HOP_SAMPLES = int(
    SR * HOP_SECONDS
)


print()
print("=" * 55)
print("MODEL SETUP COMPLETE")
print("=" * 55)

print(
    "Stage 1 parameters :",
    stage1_model.count_params()
)

print(
    "Stage 2 parameters :",
    stage2_model.count_params()
)

print()
print("Sampling Rate :", SR, "Hz")
print("Window        :", WINDOW_SECONDS, "seconds")
print("Hop           :", HOP_SECONDS, "seconds")
print("FFT Size      :", N_FFT)
print("Mel Bands     :", N_MELS)
print("CNN Input     : 128 × 130 × 1")

print()
print("Testing data:")
print(TESTING_PATH)

print()
print("✓ Ready for audio input")

Checking model files...
✓ Stage 1 model found
✓ Stage 2 model found

MODEL SETUP COMPLETE
Stage 1 parameters : 306155
Stage 2 parameters : 306155

Sampling Rate : 22050 Hz
Window        : 3.0 seconds
Hop           : 1.5 seconds
FFT Size      : 2048
Mel Bands     : 128
CNN Input     : 128 × 130 × 1

Testing data:
E:\Musical Instrument Recognition\IRMAS-TestingData

✓ Ready for audio input


In [2]:
demo_file = r"E:\Musical Instrument Recognition\IRMAS-TestingData\01 - Chet Baker - Prayer For The Newborn-8.wav"


if not os.path.exists(demo_file):
    raise FileNotFoundError(
        f"Audio file not found:\n{demo_file}"
    )


txt_file = os.path.splitext(
    demo_file
)[0] + ".txt"


if not os.path.exists(txt_file):
    raise FileNotFoundError(
        f"Ground-truth file not found:\n{txt_file}"
    )


with open(
    txt_file,
    "r",
    encoding="utf-8"
) as f:

    true_labels = [
        line.strip()
        for line in f
        if line.strip()
    ]


y, sr = librosa.load(
    demo_file,
    sr=22050,
    mono=True
)


duration = len(y) / sr

window_seconds = 3.0
hop_seconds = 1.5

window_samples = int(
    window_seconds * sr
)

hop_samples = int(
    hop_seconds * sr
)


segments = []
window_times = []

start = 0

while start < len(y):

    end = start + window_samples

    segment = y[start:end]

    actual_start = start / sr
    actual_end = min(
        end / sr,
        duration
    )

    if len(segment) == 0:
        break

    if len(segment) < window_samples:

        segment = np.pad(
            segment,
            (
                0,
                window_samples - len(segment)
            )
        )

    mel = librosa.feature.melspectrogram(
        y=segment,
        sr=sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128,
        window="hann"
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    mel_normalized = (
        mel_db + 80.0
    ) / 80.0

    segments.append(
        mel_normalized.astype(
            np.float32
        )
    )

    window_times.append(
        (
            actual_start,
            actual_end
        )
    )

    if end >= len(y):
        break

    start += hop_samples


X = np.array(
    segments,
    dtype=np.float32
)

X = X[..., np.newaxis]


print("=" * 70)
print("AUDIO TESTING")
print("=" * 70)

print(
    "File:",
    os.path.basename(demo_file)
)

print(
    f"Duration: {duration:.2f} seconds"
)

print(
    "Number of windows:",
    len(segments)
)

print(
    "CNN input shape:",
    X.shape
)

print("\nIRMAS Ground Truth:")

for label in true_labels:

    if label in instrument_names:

        print(
            f"  {instrument_names[label]}"
        )


display(
    Audio(demo_file)
)


print("\n")
print("=" * 70)
print("STAGE 1 — SINGLE-INSTRUMENT RECOGNITION")
print("=" * 70)


stage1_probabilities = stage1_model.predict(
    X,
    verbose=0
)


stage1_mean_probabilities = np.mean(
    stage1_probabilities,
    axis=0
)


stage1_index = np.argmax(
    stage1_mean_probabilities
)


stage1_label = index_to_label[
    stage1_index
]


stage1_name = instrument_names[
    stage1_label
]


stage1_confidence = (
    stage1_mean_probabilities[
        stage1_index
    ]
)


print(
    "\nPredicted Instrument:",
    stage1_name
)

print(
    f"Confidence: "
    f"{stage1_confidence:.2%}"
)


stage1_correct = (
    stage1_label in set(true_labels)
)


print(
    "\nStage 1 Result:",
    "CORRECT"
    if stage1_correct
    else
    "INCORRECT"
)


print("\nStage 1 Class Probabilities:")


stage1_sorted = sorted(
    zip(
        class_names,
        stage1_mean_probabilities
    ),
    key=lambda x: x[1],
    reverse=True
)


for label, probability in stage1_sorted:

    print(
        f"{instrument_names[label]:<20}"
        f"{probability:.2%}"
    )


print("\n")
print("=" * 70)
print("STAGE 2 — MULTI-INSTRUMENT RECOGNITION")
print("=" * 70)


stage2_probabilities = stage2_model.predict(
    X,
    verbose=0
)


stage2_mean_probabilities = np.mean(
    stage2_probabilities,
    axis=0
)


stage2_predictions = []


for class_index, probability in enumerate(
    stage2_mean_probabilities
):

    label = index_to_label[
        class_index
    ]

    threshold = final_thresholds[
        label
    ]

    if probability >= threshold:

        stage2_predictions.append(
            {
                "instrument": label,
                "confidence": float(
                    probability
                ),
                "threshold": threshold
            }
        )


stage2_predictions.sort(
    key=lambda x: x["confidence"],
    reverse=True
)


print("\nPredicted Instruments:")


if stage2_predictions:

    for prediction in stage2_predictions:

        print(
            f"{instrument_names[prediction['instrument']]:<20}"
            f"{prediction['confidence']:.2%}"
            f"  "
            f"(threshold: "
            f"{prediction['threshold']:.3f})"
        )

else:

    print(
        "No instrument crossed its threshold."
    )


actual_set = set(true_labels)

predicted_set = {
    prediction["instrument"]
    for prediction in stage2_predictions
}


correct_labels = (
    actual_set &
    predicted_set
)


missed_labels = (
    actual_set -
    predicted_set
)


extra_labels = (
    predicted_set -
    actual_set
)


if actual_set == predicted_set:

    stage2_result = "CORRECT"

elif correct_labels:

    stage2_result = "PARTIALLY CORRECT"

else:

    stage2_result = "INCORRECT"


print(
    "\nStage 2 Result:",
    stage2_result
)


if correct_labels:

    print("\nCorrectly Detected:")

    for label in sorted(
        correct_labels
    ):

        print(
            f"  ✓ "
            f"{instrument_names[label]}"
        )


if missed_labels:

    print("\nMissed Instruments:")

    for label in sorted(
        missed_labels
    ):

        print(
            f"  ✗ "
            f"{instrument_names[label]}"
        )


if extra_labels:

    print("\nFalse Positive Instruments:")

    for label in sorted(
        extra_labels
    ):

        print(
            f"  ✗ "
            f"{instrument_names[label]}"
        )


print("\nStage 2 Class Probabilities:")


stage2_sorted = sorted(
    zip(
        class_names,
        stage2_mean_probabilities
    ),
    key=lambda x: x[1],
    reverse=True
)


for label, probability in stage2_sorted:

    threshold = final_thresholds[
        label
    ]

    status = (
        "DETECTED"
        if probability >= threshold
        else ""
    )

    print(
        f"{instrument_names[label]:<20}"
        f"{probability:.2%} "
        f"(threshold: {threshold:.3f}) "
        f"{status}"
    )


print("\n")
print("=" * 70)
print("WINDOW-WISE STAGE 2 PREDICTIONS")
print("=" * 70)


for i, (
    start_time,
    end_time
) in enumerate(window_times):

    predictions = []

    for class_index, probability in enumerate(
        stage2_probabilities[i]
    ):

        label = index_to_label[
            class_index
        ]

        threshold = final_thresholds[
            label
        ]

        if probability >= threshold:

            predictions.append(
                (
                    label,
                    probability
                )
            )


    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )


    if predictions:

        prediction_text = ", ".join(
            [
                f"{instrument_names[label]} "
                f"({probability:.1%})"
                for label, probability
                in predictions
            ]
        )

    else:

        prediction_text = "None"


    print(
        f"{start_time:.1f}-{end_time:.1f} s"
        f" : "
        f"{prediction_text}"
    )


print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)


print(
    "\nActual:",
    ", ".join(
        instrument_names[label]
        for label in true_labels
        if label in instrument_names
    )
)


print(
    f"\nStage 1 Prediction: "
    f"{stage1_name} "
    f"({stage1_confidence:.2%})"
)

print(
    "Stage 1:",
    "CORRECT"
    if stage1_correct
    else
    "INCORRECT"
)


if stage2_predictions:

    stage2_text = ", ".join(
        instrument_names[
            prediction["instrument"]
        ]
        for prediction in stage2_predictions
    )

else:

    stage2_text = "None"


print(
    "\nStage 2 Prediction:",
    stage2_text
)

print(
    "Stage 2:",
    stage2_result
)


print("\n" + "=" * 70)

AUDIO TESTING
File: 01 - Chet Baker - Prayer For The Newborn-8.wav
Duration: 20.00 seconds
Number of windows: 13
CNN input shape: (13, 128, 130, 1)

IRMAS Ground Truth:
  Piano
  Trumpet




STAGE 1 — SINGLE-INSTRUMENT RECOGNITION

Predicted Instrument: Trumpet
Confidence: 40.53%

Stage 1 Result: CORRECT

Stage 1 Class Probabilities:
Trumpet             40.53%
Saxophone           24.93%
Clarinet            13.13%
Piano               8.00%
Violin              6.77%
Flute               3.78%
Electric Guitar     1.44%
Acoustic Guitar     0.66%
Cello               0.62%
Organ               0.11%
Voice               0.01%


STAGE 2 — MULTI-INSTRUMENT RECOGNITION

Predicted Instruments:
Trumpet             47.06%  (threshold: 0.200)

Stage 2 Result: PARTIALLY CORRECT

Correctly Detected:
  ✓ Trumpet

Missed Instruments:
  ✗ Piano

Stage 2 Class Probabilities:
Trumpet             47.06% (threshold: 0.200) DETECTED
Clarinet            33.48% (threshold: 0.350) 
Saxophone           21.36% (threshold: 0.225) 
Flute               11.43% (threshold: 0.400) 
Piano               4.41% (threshold: 0.125) 
Violin              1.28% (threshold: 0.175) 
Cello               1.04% (threshol